# Valhalla Isochrones - Silver Layer

Generates 5-minute driving isochrones for existing LCE stores using Valhalla API.

**API Reference:** https://valhalla.github.io/valhalla/api/isochrone/api-reference/

**Input:**
- `{catalog}.{bronze_schema}.lce_locations_mass`

**Output:**
- `{catalog}.{silver_schema}.isochrones_lce_valhalla`

## Parameters

In [ ]:
import requests
import json
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

dbutils.widgets.text("catalog", "jdub_demo_aws")
dbutils.widgets.text("bronze_schema", "geo_bronze")
dbutils.widgets.text("silver_schema", "geo_silver")
dbutils.widgets.text("lce_locations_table", "lce_locations_mass")
dbutils.widgets.text("valhalla_url", "https://valhalla1.openstreetmap.de")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
lce_locations_table = dbutils.widgets.get("lce_locations_table")
valhalla_url = dbutils.widgets.get("valhalla_url")

input_table = f"{catalog}.{bronze_schema}.{lce_locations_table}"
output_table = f"{catalog}.{silver_schema}.isochrones_lce_valhalla"

print(f"Input: {input_table}")
print(f"Output: {output_table}")
print(f"Valhalla URL: {valhalla_url}")

## Test Valhalla Connection

In [ ]:
# Test Valhalla API with a sample location (Boston)
test_payload = {
    "locations": [{"lat": 42.3601, "lon": -71.0589}],
    "costing": "auto",
    "contours": [{"time": 5}],  # 5 minutes
    "polygons": True
}

test_url = f"{valhalla_url}/isochrone"

try:
    response = requests.post(test_url, json=test_payload, timeout=30)
    response.raise_for_status()
    test_result = response.json()
    
    print("✓ Successfully connected to Valhalla API")
    print(f"  Test response features: {len(test_result.get('features', []))}")
    print(f"  Response type: {test_result.get('type')}")
except Exception as e:
    print(f"❌ Could not connect to Valhalla API: {e}")
    raise

## Load LCE Store Locations

In [ ]:
# Load store locations
stores = spark.table(input_table)

# Standardize column names
columns = stores.columns
id_col = next((c for c in columns if c in ['store_number', 'id', 'location_id']), columns[0])
lat_col = next((c for c in columns if c in ['latitude', 'lat', 'y']), None)
lon_col = next((c for c in columns if c in ['longitude', 'lon', 'lng', 'x']), None)
city_col = next((c for c in columns if c in ['city', 'municipality']), None)
state_col = next((c for c in columns if c in ['state', 'region', 'state_abbr']), None)

if not lat_col or not lon_col:
    raise ValueError(f"Cannot find lat/lon columns. Available: {columns}")

stores_clean = stores.select(
    F.col(id_col).alias("store_number"),
    F.col(lat_col).cast("double").alias("latitude"),
    F.col(lon_col).cast("double").alias("longitude"),
    F.col(city_col).alias("city") if city_col else F.lit(None).alias("city"),
    F.col(state_col).alias("state") if state_col else F.lit(None).alias("state")
).filter(
    F.col("latitude").isNotNull() & F.col("longitude").isNotNull()
)

print(f"Loaded {stores_clean.count()} LCE store locations")
display(stores_clean.limit(5))

## Generate Isochrones with Valhalla

In [ ]:
def get_valhalla_isochrone(lat, lon, minutes, valhalla_url):
    """
    Get isochrone from Valhalla API
    
    Returns: GeoJSON polygon as WKT string
    """
    payload = {
        "locations": [{"lat": lat, "lon": lon}],
        "costing": "auto",
        "contours": [{"time": minutes}],
        "polygons": True
    }
    
    try:
        response = requests.post(
            f"{valhalla_url}/isochrone",
            json=payload,
            timeout=30
        )
        response.raise_for_status()
        result = response.json()
        
        # Extract polygon from GeoJSON
        if result.get('features') and len(result['features']) > 0:
            feature = result['features'][0]
            geometry = feature.get('geometry')
            
            if geometry and geometry.get('type') == 'Polygon':
                coords = geometry['coordinates'][0]  # Exterior ring
                # Convert to WKT format: POLYGON ((lon lat, lon lat, ...))
                coords_str = ', '.join([f"{lon} {lat}" for lon, lat in coords])
                return f"POLYGON (({coords_str}))"
        
        return None
        
    except Exception as e:
        print(f"Error for location ({lat}, {lon}): {e}")
        return None

In [ ]:
# Test with first store
test_store = stores_clean.first()
print(f"Testing with store: {test_store.store_number}")
print(f"Location: ({test_store.latitude}, {test_store.longitude})")

test_wkt = get_valhalla_isochrone(
    test_store.latitude, 
    test_store.longitude, 
    5, 
    valhalla_url
)

if test_wkt:
    print("✓ Test isochrone generated successfully!")
    print(f"  WKT length: {len(test_wkt)} characters")
else:
    print("⚠ Test failed - no isochrone generated")

## Generate All Isochrones

In [ ]:
from pyspark.sql import Row
import time

store_rows = stores_clean.collect()
print(f"Generating 5-minute isochrones for {len(store_rows)} stores...\n")

results = []
start_time = time.time()

for i, store in enumerate(store_rows):
    if i % 5 == 0 and i > 0:
        elapsed = time.time() - start_time
        avg_time = elapsed / i
        remaining = (len(store_rows) - i) * avg_time
        print(f"Progress: {i}/{len(store_rows)} ({i/len(store_rows)*100:.1f}%) - "
              f"Elapsed: {elapsed:.1f}s - ETA: {remaining:.1f}s")
    
    wkt = get_valhalla_isochrone(
        store.latitude,
        store.longitude,
        5,
        valhalla_url
    )
    
    if wkt:
        results.append(Row(
            store_number=store.store_number,
            latitude=store.latitude,
            longitude=store.longitude,
            city=store.city,
            state=store.state,
            drive_time_minutes=5,
            geometry_wkt=wkt
        ))
    
    # Small delay to be respectful of the API
    time.sleep(0.1)

total_time = time.time() - start_time
print(f"\n✅ Generated {len(results)}/{len(store_rows)} isochrones")
print(f"   Total time: {total_time:.1f} seconds")
print(f"   Average: {total_time/len(store_rows):.2f} seconds per store")

## Save to Delta

In [ ]:
# Create DataFrame
schema = StructType([
    StructField("store_number", StringType(), False),
    StructField("latitude", DoubleType(), False),
    StructField("longitude", DoubleType(), False),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("drive_time_minutes", IntegerType(), False),
    StructField("geometry_wkt", StringType(), False)
])

isochrones_df = spark.createDataFrame(results, schema=schema)

# Convert to geometry and add metadata
isochrones_final = (
    isochrones_df
    .withColumn("geometry", F.expr("ST_GeomFromText(geometry_wkt, 4326)"))
    .withColumn("area_sqkm", F.expr("ST_Area(geometry) / 1000000"))
    .withColumn("created_timestamp", F.current_timestamp())
    .withColumn("routing_provider", F.lit("valhalla"))
    .withColumn("store_type", F.lit("Little Caesars"))
    .drop("geometry_wkt")
)

# Write to Delta
(
    isochrones_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(output_table)
)

print(f"Saved {len(results)} isochrones to {output_table}")

## Visualize with Folium

In [ ]:
import folium
from shapely import wkt as shapely_wkt

# Create map centered on Massachusetts
ma_center = [42.4072, -71.3824]
m = folium.Map(location=ma_center, zoom_start=8, tiles='OpenStreetMap')

# Add isochrone polygons
print(f"Adding {len(results)} isochrone polygons...")
for result in results:
    polygon = shapely_wkt.loads(result.geometry_wkt)
    coords = [[lat, lon] for lon, lat in polygon.exterior.coords]
    
    folium.Polygon(
        locations=coords,
        color='#FF6B35',
        fillColor='#FF6B35',
        fillOpacity=0.2,
        weight=2,
        popup=f"Store: {result.store_number}<br>5 min drive time (Valhalla)"
    ).add_to(m)

# Add store location markers
print(f"Adding {len(store_rows)} store markers...")
for store in store_rows:
    folium.CircleMarker(
        location=[store.latitude, store.longitude],
        radius=6,
        popup=f"<b>Little Caesars</b><br>Store: {store.store_number}<br>{store.city}, {store.state}",
        color='#C1121F',
        fillColor='#C1121F',
        fillOpacity=0.8,
        weight=2
    ).add_to(m)

# Add legend
legend_html = '''
<div style="position: fixed; 
            bottom: 50px; right: 50px; width: 200px; height: 100px; 
            background-color: white; border:2px solid grey; z-index:9999; 
            font-size:14px; padding: 10px">
<p style="margin-bottom: 5px;"><b>Legend</b></p>
<p style="margin: 5px 0;"><span style="color: #C1121F;">●</span> LCE Store</p>
<p style="margin: 5px 0;"><span style="background-color: rgba(255,107,53,0.3); padding: 0 8px;">█</span> 5-min Drive (Valhalla)</p>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

print(f"\n✅ Map created with {len(results)} isochrones and {len(store_rows)} stores")
m

## Summary Statistics

In [ ]:
print("Valhalla Isochrone Summary:")
display(spark.sql(f"""
    SELECT
        COUNT(*) as total_isochrones,
        ROUND(AVG(area_sqkm), 2) as avg_area_sqkm,
        ROUND(MIN(area_sqkm), 2) as min_area_sqkm,
        ROUND(MAX(area_sqkm), 2) as max_area_sqkm,
        routing_provider,
        drive_time_minutes
    FROM {output_table}
    GROUP BY routing_provider, drive_time_minutes
"""))